## Install Dependencies

In [ ]:
!pip install langchain transformers torch streamlit wikipedia pyngrok PyPDF2 sentence-transformers faiss-cpu langchain-huggingface langchain-community

## 1.Import Libs

In [ ]:
import os
import streamlit as st
import torch
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline

## 2. Memory Optimization
To prevent CUDA memory issues, set PyTorch’s memory management and clear the GPU cache:

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

## 3. PDF Reading
The `get_pdf_text` function extracts text from uploaded PDFs, iterating through each page and concatenating the text into a single string

In [1]:
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        pdf_reader = PdfReader(pdf)
        for page in pdf_reader.pages:
            extracted_text = page.extract_text()
            if extracted_text:
                text += extracted_text + "\n"
    return text

## 4. Text Chunking
The `get_text_chunks` function splits the extracted text into smaller chunks (1000 characters, 200-character overlap) to maintain context for embeddings.

In [ ]:
def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    return text_splitter.split_text(text)

## 5. Embedding Generation and Vector Store Creation
The `get_vectorstore` function embeds text chunks using `sentence-transformers/all-MiniLM-L6-v2` and stores them in a FAISS vector database for similarity search.

In [ ]:
def get_vectorstore(text_chunks):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        return FAISS.from_texts(texts=text_chunks, embedding=embeddings)
    except Exception as e:
        st.error(f"Error creating vector store: {e}")
        return None

## 6. Conversation Chain Setup (LLM, Retriever, Memory)
The `get_conversation_chain` function:

- Loads `google/flan-t5-small` locally with 16-bit precision to save memory.
- Creates a Hugging Face pipeline for LangChain.
- Sets up a memory buffer and conversational retrieval chain to answer questions based on the vector store and chat history.

In [ ]:
def get_conversation_chain(vectorstore):
    try:
        # Load model and tokenizer
        tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", torch_dtype=torch.float16)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        st.write(f"Model loaded on {device}")

        # Create pipeline
        pipe = pipeline(
            "text2text-generation",
            model=model,
            tokenizer=tokenizer,
            max_length=256,
            temperature=0.7,
            device=0 if torch.cuda.is_available() else -1
        )
        llm = HuggingFacePipeline(pipeline=pipe)

        memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
        return ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=vectorstore.as_retriever(),
            memory=memory
        )
    except Exception as e:
        st.error(f"Error setting up conversation chain: {e}")
        return None

## 7. User Input Handler (Ask Questions and Display Chat)
The `handle_userinput` function processes user questions, checks if the conversation chain is ready, and displays the chat history in Streamlit.

In [ ]:
def handle_userinput(user_question):
    if "conversation" not in st.session_state or not st.session_state.conversation:
        st.error("Please upload and process a PDF first.")
        return
    try:
        response = st.session_state.conversation({'question': user_question})
        st.session_state.chat_history = response['chat_history']
        st.subheader("Chat History")
        for i, message in enumerate(st.session_state.chat_history):
            speaker = "You:" if i % 2 == 0 else "AI:"
            st.write(f"**{speaker}** {message.content}")
    except Exception as e:
        st.error(f"Error processing question: {e}")

## 8. User Interface (Streamlit App Logic)
The `main` function sets up the Streamlit app, handles PDF uploads, processes text, and manages user interactions. Save this as `app.py`:

In [ ]:
%%writefile app.py
import os
import torch
import streamlit as st
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline

# Memory optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

# Set the title using Streamlit
st.title('PDF Summarizer Bot')

# PDF text extraction
def get_pdf_text(pdf_docs):
    text = ""
    try:
        for pdf in pdf_docs:
            pdf_reader = PdfReader(pdf)
            for page in pdf_reader.pages:
                extracted_text = page.extract_text()
                if extracted_text:
                    text += extracted_text + "\n"
        return text
    except Exception as e:
        st.error(f"Error extracting PDF text: {e}")
        return ""

# Text chunking
def get_text_chunks(text):
    try:
        text_splitter = CharacterTextSplitter(
            separator="\n",
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len
        )
        return text_splitter.split_text(text)
    except Exception as e:
        st.error(f"Error chunking text: {e}")
        return []

# Vector store creation
def get_vectorstore(text_chunks):
    try:
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        return FAISS.from_texts(texts=text_chunks, embedding=embeddings)
    except Exception as e:
        st.error(f"Error creating vector store: {e}")
        return None

# Conversation chain setup
def get_conversation_chain(vectorstore):
    try:
        tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
        model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", torch_dtype=torch.float16)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)
        st.write(f"Model loaded on {device}")

        pipe = pipeline(
            "text2text-generation",
            model=model,
            tokenizer=tokenizer,
            max_length=256,
            temperature=0.7,
            device=0 if torch.cuda.is_available() else -1
        )
        llm = HuggingFacePipeline(pipeline=pipe)

        memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
        return ConversationalRetrievalChain.from_llm(
            llm=llm,
            retriever=vectorstore.as_retriever(),
            memory=memory
        )
    except Exception as e:
        st.error(f"Error setting up conversation chain: {e}")
        return None

# Handle user input
def handle_userinput(user_question):
    if "conversation" not in st.session_state or not st.session_state.conversation:
        st.error("Please upload and process a PDF first.")
        return
    try:
        response = st.session_state.conversation({'question': user_question})
        st.session_state.chat_history = response['chat_history']
        st.subheader("Chat History")
        for i, message in enumerate(st.session_state.chat_history):
            speaker = "You:" if i % 2 == 0 else "AI:"
            st.write(f"**{speaker}** {message.content}")
    except Exception as e:
        st.error(f"Error processing question: {e}")

# Main Streamlit app
def main():
    st.set_page_config(page_title="Brainstorm with PDFs", page_icon=None)
    st.title("Brainstorm with Your PDFs")

    if "conversation" not in st.session_state:
        st.session_state.conversation = None
    if "chat_history" not in st.session_state:
        st.session_state.chat_history = []

    with st.sidebar:
        st.subheader("Upload Your Documents")
        pdf_docs = st.file_uploader("Upload PDFs", accept_multiple_files=True, type=["pdf"])
        if st.button("Process"):
            with st.spinner("Processing PDFs..."):
                raw_text = get_pdf_text(pdf_docs)
                if not raw_text:
                    st.error("No text extracted from PDFs.")
                    return
                text_chunks = get_text_chunks(raw_text)
                if not text_chunks:
                    st.error("No text chunks created.")
                    return
                vectorstore = get_vectorstore(text_chunks)
                if vectorstore:
                    st.session_state.conversation = get_conversation_chain(vectorstore)
                    st.success("Processing complete! You can now chat with your PDFs.")

    user_question = st.chat_input("Ask a question about your PDFs:")
    if user_question:
        handle_userinput(user_question)

if __name__ == '__main__':
    main()

## Access the App via NGROK

In [ ]:
from pyngrok import ngrok

# Replace "YOUR_AUTHTOKEN" with your actual ngrok authtoken
ngrok.set_auth_token("YOUR_AUTHTOKEN")

public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py --server.port 8501 &
public_url = ngrok.connect(8501)
print(public_url)